# Quantum Agent AI — Google Colab

รันทุก step ทีละ cell (กด Shift+Enter)

**โหมดที่มี:**
- Standard — แชทปกติ
- Quantum — คิดหลายมุม
- Ternary — ตอบ -1/0/1
- Fusion — หลอมรวมทุกอย่าง
- Knowledge Memory — จำข้อมูลถาวร

## Step 1: ติดตั้ง Quantum Agent + Ollama

In [ ]:
# ติดตั้ง quantum-agent framework
!pip install -q git+https://github.com/tanayut0108-source/ai-agent-quantum.git@devin/1779634602-chat-conversation-feature

# ติดตั้ง Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n=== ติดตั้งเสร็จ! ===")

## Step 2: เปิด Ollama Server + โหลด Model

In [ ]:
import subprocess
import time

# เปิด Ollama server ใน background
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
print("Ollama server started!")

# โหลด model (เลือก 1 ตัว — uncomment ที่ต้องการ)
# TinyLlama (เล็ก เร็ว ~600MB)
!ollama pull tinyllama

# Gemma 2 9B (ฉลาดมาก ~5.5GB — ต้องใช้ GPU)
# !ollama pull gemma2:9b

# Qwen 2.5 7B (ภาษาไทยดี ~4.7GB)
# !ollama pull qwen2.5:7b

print("\n=== Model พร้อมใช้งาน! ===")

## Step 3: ทดสอบ — Standard Mode (แชทปกติ)

In [ ]:
from quantum_agent.llm import OllamaBackend
from quantum_agent.chat import ChatSession

# โหลด model (เปลี่ยนชื่อตาม model ที่โหลดใน Step 2)
llm = OllamaBackend(model="tinyllama")
llm.load()
print(f"Model: {llm.model_info().name}")

# Standard chat
session = ChatSession(llm)
reply = session.send("What is quantum computing?")
print(f"\nAgent: {reply.content}")

## Step 4: ทดสอบ — Quantum Mode (คิดหลายมุม)

In [ ]:
# Quantum mode — สร้าง hypotheses หลายทาง แล้วรวมเป็นคำตอบ
session_q = ChatSession(llm, quantum_mode=True)
reply = session_q.send("How to build a web scraper?")
print(f"Agent (Quantum): {reply.content}")

## Step 5: ทดสอบ — Ternary Mode (ตอบ -1/0/1)

In [ ]:
# Ternary mode — ตอบ -1 (ไม่), 0 (ไม่แน่นอน), 1 (ใช่)
session_t = ChatSession(llm, ternary_mode=True)

questions = [
    "Is the earth round?",
    "Can humans fly without machines?",
    "Do aliens exist?",
    "Is water wet?",
    "Will AI take over the world?",
]

for q in questions:
    reply = session_t.send(q)
    label = {"1": "YES", "0": "UNCERTAIN", "-1": "NO"}.get(reply.content, "?")
    print(f"{q:45s} => {reply.content} ({label})")

## Step 6: ทดสอบ — Knowledge Memory (จำข้อมูลถาวร)

In [ ]:
from quantum_agent.chat import KnowledgeStore

# สร้าง memory store
knowledge = KnowledgeStore("/content/brain.json")

# สร้าง session ที่มี memory
session_m = ChatSession(llm, knowledge=knowledge)

# สอน AI
print(session_m.remember("Python was created by Guido van Rossum in 1991"))
print(session_m.remember("The speed of light is 299,792,458 meters per second"))
print(session_m.remember("Bangkok is the capital of Thailand"))

# ดู memories
print("\nMemories:")
for m in session_m.list_memories():
    print(f"  {m}")

# ถามคำถามที่เกี่ยวกับ memory
reply = session_m.send("Who created Python?")
print(f"\nAgent: {reply.content}")

## Step 7: ทดสอบ — Fusion Mode (หลอมรวมทุกอย่าง)

In [ ]:
# Fusion mode — รวม Quantum + Ternary + Memory
session_f = ChatSession(llm, fusion_mode=True, knowledge=knowledge)

# คำถาม yes/no — จะได้ verdict
print("=== Yes/No Question ===")
reply = session_f.send("Is Python a good language for beginners?")
print(f"Agent: {reply.content}")

print("\n=== Open Question ===")
reply = session_f.send("Explain how to learn programming")
print(f"Agent: {reply.content}")

## Step 8: ทดสอบ — Quantum Core (สำหรับนักพัฒนา)

In [ ]:
from quantum_agent.core import QubitParameter, ParameterRegister, TernaryValue
from quantum_agent.core.geodesic_mesh import GeodesicMesh3D
import numpy as np

# Qubit Parameters
qp = QubitParameter("confidence", value=0.8)
print(f"QubitParameter: {qp}")
print(f"  Amplitudes: {qp.amplitudes}")
print(f"  Probabilities: {qp.probabilities}")

# Parameter Register
reg = ParameterRegister()
reg.set("alpha", 0.5)
reg.set("beta", -0.3)
reg.set("gamma", 0.9)
print(f"\nRegister: {reg}")

# Ternary Logic
print(f"\nTernary: YES={TernaryValue.YES}, NO={TernaryValue.NO}, MAYBE={TernaryValue.MAYBE}")

# Geodesic Mesh 3D
mesh = GeodesicMesh3D(subdivisions=2, n_shells=10)
print(f"\nGeodesicMesh3D: {mesh.vertex_count} vertices x {mesh.n_shells} shells")
print(f"  Total parameters: {mesh.vertex_count * mesh.n_shells:,}")

## Step 9: Interactive Chat (แชทแบบโต้ตอบ)

รัน cell นี้แล้วพิมพ์คุยกับ AI ได้เลย!

In [ ]:
# Interactive chat loop
session_interactive = ChatSession(
    llm,
    fusion_mode=True,
    knowledge=knowledge,
)

print("Quantum Agent Chat (Fusion Mode)")
print("พิมพ์ 'quit' เพื่อออก")
print("-" * 40)

while True:
    try:
        user = input("\nYou: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nBye!")
        break

    if not user or user.lower() in ("quit", "exit"):
        print("Bye!")
        break

    if user.startswith("/remember "):
        print(session_interactive.remember(user[10:]))
        continue

    if user == "/memories":
        for m in session_interactive.list_memories():
            print(f"  {m}")
        continue

    reply = session_interactive.send(user)
    print(f"\nAgent: {reply.content}")